In [2]:
# Import packages that will be used throughout the notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import requests
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
import plotly.express as px

# Problem 1: Unconstrained IK

This problem walks you through the process of performing unconstrained and constrained inverse kinematics (IK) given real experimental motion capture data from a research study. By the end of this homework, you will understand and implement:

1. The general workflow for **unconstrained inverse kinematics** (determining joint angles based on time-varying anatomical marker locations). This method is deemed "unconstrained" because each segment is assumed to have 6 degrees of freedom (6-DOF), with no defined joints or range of motion limits. Consequently, calculated rigid body segment lengths may vary over time, and joint constraints may be violated (i.e. segments may "dislocate") due to unavoidable experimental issues like soft-tissue artifact (STA).
    - The "Error" in unconstrained IK looks like kinematic inconsistency. Segment lengths may vary in time and discontinuities at the joints (spatial gaps between adjacent segments).

2. The general workflow for **constrained inverse kinematics** (estimating the pose of a multi-segment body by minimizing a weighted least squares error between predicted marker locations - via forward kinematics - and measured experimental marker locations at each time step). This method is "constrained" because joints are defined within a generic musculoskeletal model, which governs the possible motion of adjacent segments. Scaling is performed prior to the optimization routine to fix segment lengths so they do not vary in time. Additionally, physiological range of motion (ROM) limits are often prescribed for the defined joints.
    - The "Error" in Constrained IK looks like marker residuals. Since the model's rigid geometry cannot perfectly match the experimental markers, the error is the squared distance (RMSE) between where the model says a marker should be and where the markers were experimentally observed.

## 1a: Read in the gait data

26 passive reflective markers were placed on a neurotypical subject before they walked on a dual-belt instrumented treadmill surrounded by 10 Vicon motion capture cameras recording at 100 Hz for approximately 5 minutes. (Note: For this analysis, we will be using a subset of 24 markers). If you are interested in more details of the study, check out the paper:

- Bacek, Tomislav (2024). Biomechanics and energetics of neurotypical (un)constrained walking [Bacek2023]. The University of Melbourne. Collection. https://doi.org/10.26188/c.6887854.v1

Commercial motion capture systems output data in a variety of formats. In this case, we have converted the .c3d files generated by the Vicon Nexus 2.8 software into a .trc (Track Row Column) file format. The .trc file is a tab-separated text file containing time-series kinematic data and metadata (e.g., frame rate, number of markers). We begin by reading in the motion capture data. Below are the header rows and the first few data rows for the first three markers (RASI, RPSI, and LPSI).

| PathFileType | 4 | (X/Y/Z) | sub7_Kinematics_T1.trc |   |   |   |   |   |   |   |  … |
|--------|----------|------|------|------|------|------|------|------|------|------|---|
| DataRate | CameraRate | NumFrames | NumMarkers | Units | OrigDataRate | OrigDataStartFrame | OrigNumFrames |  |  |  | … |
| 100 | 100 | 30049 | 24 | mm | 100 | 1 | 102 |  |  |  | … |
| Frame# | Time (s) | RASI |  |  | RPSI |  |  | LPSI |  |  | … |
|        |          | X1 | Y1 | Z1 | X2 | Y2 | Z2 | X3 | Y3 | Z3 | … |
| 1 | 0.00 | -175.398 | 58.105 | 939.640 | -45.887 | 215.503 | 967.576 | 12.043 | 208.794 | 966.499 | … |
| 2 | 0.01 | -176.398 | 59.754 | 940.343 | -47.041 | 217.132 | 969.229 | 11.059 | 210.466 | 968.183 | … |
| 3 | 0.02 | -177.152 | 61.618 | 941.166 | -47.898 | 218.504 | 970.631 | 10.028 | 211.998 | 969.691 | … |


To simplify the analysis, we restrict ourselves to **sagittal-plane motion**. In the original trial, the subject walks in the $-Y$ direction, with the $+Z$ axis as the superior (upward) direction. For ease of visualization, we will invert the Y-axis coordinates so that the subject walks in the $+Y$ direction.

<img src="https://uofi.box.com/shared/static/c78q03kuoxrwfxe2m98nm6ctdk1ar60b.png" alt="Gait cycle def with coord sys" width="900"/>

There is nothing you need to change in cell 1a. Just skim the function docstrings and few lines of code used to load and filter the data before running the cell.

In [ ]:
# Create a helper function to read in a file from a URL
def download_file(url, save_path):
    """
    Downloads a file from a given URL and saves it to a specified path.

    Args:
        url (str): The URL of the file to download.
        save_path (str): The local path where the file will be saved.

    Returns:
        str: The path where the file was saved.

    Raises:
        requests.exceptions.RequestException: If an error occurs during the download.
    """
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(save_path, 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
    return save_path

# Create helper function to read in a trc file containing motion capture marker data and only keep markers relevant for this problem.
def read_trc_file(filepath) -> pd.DataFrame:
    
    """
    Reads a trc file and returns a pandas DataFrame with properly named columns

    Parameters:
    -----------
    filepath (Str): The path to the trc file to be read

    Returns:
    --------
    downsampled_data (pandas.DataFrame): A DataFrame containing all marker data with columns named as 'MarkerName_X', 'MarkerName_Y', 'MarkerName_Z'

    """

    # Read the file lines to extract marker names
    with open(filepath, 'r') as f:
        lines = f.readlines()
    
    # Row 3 has marker names (0-indexed)
    marker_line = lines[3].strip().split('\t')
    
    # Read the actual data starting in row 4
    df = pd.read_csv(filepath, delimiter='\t', skiprows=4)
    
    # Now create proper column names
    new_columns = []
    
    # First two columns are special
    new_columns.extend(['Frame', 'Time'])
    
    # Process marker columns in groups of 3 (X,Y,Z)
    marker_index = 0
    for i in range(2, len(df.columns), 3):
        if i + 2 < len(df.columns):
            # Get marker name (skip first two columns which contain Frame#, Time)
            marker_name = marker_line[i] if i < len(marker_line) else f"Marker{marker_index+1}"
            
            # Add X, Y, Z columns for this marker
            new_columns.extend([f"{marker_name}_X", f"{marker_name}_Y", f"{marker_name}_Z"])
            marker_index += 1
    
    # Only assign as many columns as we have markers
    df.columns = new_columns[:len(df.columns)]

    # Define which markers we want to keep from the raw data
    markers_to_keep = ['RHIP_Y', 'RHIP_Z', 'RLKN_Y', 'RLKN_Z', 'RLM_Y', 'RLM_Z', 'RTOE_Y', 'RTOE_Z','RCAL_Y','RCAL_Z']

    # Add frame number and time
    columns_to_keep = ['Frame', 'Time'] + markers_to_keep

    downsampled_data = df[columns_to_keep].copy()

    # reflect the markers across the Z axis
    for marker in markers_to_keep:
        if marker.endswith(('_Y')):
            downsampled_data[marker] *= -1 
        
    return downsampled_data

gait_data_path = download_file("https://uofi.box.com/shared/static/ol0jf4r94pnaq8c3ejjndyg80sk54n9j.trc", 'gait_data.trc') # Pull the file from the URL
data = read_trc_file('gait_data.trc') # Read in the trc file

print("\nData loaded and preprocessed successfully!")

## 1b: Visualize the markers of interest for all gait cycles

When we loaded the data in the above cell, we only kept the markers placed approximately on the right hip joint center (RHIP), the lateral aspect of the right knee/ankle (RLKN/RLM), a marker placed on the right calcalenous (RCAL), and a marker placed on the dorsal aspect of the foot, specifically over the second metatarsal head, where the toe meets the foot (RTOE). 

There is nothing you need to change in cell 1b. Just run the cell and observe the sagittal-plane trace of all our markers of interest over all gait cycles. To help understand what is going on, the markers start larger and opaque and become smaller as time progresses. You will observe something that looks more like point "clouds." This is completely expected. Recall this is data for 5 minutes of walking, and these "clouds" simply indicate that the subject drifted slightly in the anterior/posterior on the treadmill over this time period. 

You should also notice that the origin (0,0) was defined during a calibration, therefore all of our marker position locations are reported relative to it. It makes sense that the Z location of the toe is fairly close to zero throughout most of the test, because the calibration origin for the reflective marker system was likley placed somewhere on/near the surface of the treadmill before collecting data.

In [ ]:
def plot_marker_trajectory(data, markers, figsize=(6,6)):
    """
    Plot Y–Z trajectories for motion capture markers with fading transparency
    and decreasing marker size over time.

    Parameters
    ----------
    data : pandas.DataFrame
        DataFrame containing marker coordinate columns (e.g., RHIP_Y, RHIP_Z)

    markers : list of str
        Marker base names (e.g., ['RHIP','RLKN','RLM','RTOE'])

    figsize : tuple
        Matplotlib figure size

    Returns
    -------
    fig, ax : matplotlib figure and axis
    """

    fig, ax = plt.subplots(figsize=figsize)

    # Create gradient transparency and size
    alphas = np.linspace(1.0, 0.1, len(data))
    sizes = np.linspace(30, 2, len(data))

    for marker in markers:
        y_data = data[f"{marker}_Y"]
        z_data = data[f"{marker}_Z"]

        ax.scatter(
            y_data,
            z_data,
            label=marker,
            alpha=alphas,
            s=sizes
        )

    ax.set_xlabel("Y location (mm)")
    ax.set_ylabel("Z location (mm)")
    ax.legend(frameon=False)
    ax.grid(True, alpha=0.3)

    return fig, ax

# Use our function to make a static maker trajectory plot
plot_marker_trajectory(data,['RHIP','RLKN','RCAL','RLM','RTOE'])
plt.show()

## 1c: Trim the data to only include one gait cycle

For this assignment, we only want to analyze one right leg gait cycle (heel strike to heel strike), but from the above plot, clearly we have **many** gait cycles. Therefore, you will need to examine the data and determine timestamps for two successive right heelstrikes.

One way to approach this would be to examine the z-coordinate of right calcaneus in the ```data``` Pandas DataFrame object that we created in step 1a. A good start would be to create an interactice plot of the z-coordinate of right calcaneus vs.time by running the cell below. You should then be able to identify two heelstrike timestamps. *Hint* You may need to zoom in to find two succesive heel strikes in the interactive plot.

In [ ]:
# Create an interactive plot of the right calcaneus vertical position (Z) over time using plotly

fig = px.line(data, x='Time', y='RCAL_Z', 
              title='Right Calcaneus Vertical Position (Z-coordinate) vs. Time',
              labels={'RCAL_Z': 'Z-Coordinate Location (mm)', 'Time': 'Time (s)'})

# Display the interactive plot
fig.show()

Now that you have idenitified two timestamps, use them alongside the helper function provided below. The helper function is completed. All you need to do is read the docstring to understand how to use it and then trim your ```data``` object from part 1a.

In [ ]:
# Create a helper function to filter the data to only include a single gait cycle
def trim_trc_data(data,t_i,t_f) -> pd.DataFrame:

    """
    Trim data from a DataFrame and returns a new DataFrame that only includes data between times t_i and t_f

    Parameters:
    -----------
    data (pandas.DataFrame): A DataFrame containing all marker data and a column named "Time" (assumed to be in seconds)
    t_i (float): The intial time used for trimming (assumed to be in seconds)
    t_f (float): The final time used for trimming (assumed to be in seconds)

    Returns:
    --------
    trimmed_data (pandas.DataFrame): A DataFrame containing all data where the "Time" column was between the user-specified intial and final times

    """

    # Trim data to only include data between the supplied inital and final times
    trimmed_data = data[(data['Time'] >= t_i) & (data['Time'] <= t_f)].reset_index(drop=True)

    return trimmed_data

In [ ]:
# Fill in the missing timestamp values, then run the cell to create your trimmed dataframe for a single right leg gait cycle
trimmed_data = trim_trc_data(data,t_i = #TODO fill in the initial timestamp you identified in part 1c , 
                             t_f = #TODO fill in the final timestamp you identified in part)

As a quick check that you have done this step correctly, run the cell below (there is nothing you need to change) and you should see a static plot of the hip, knee, ankle, and toe markers over a **single gait cycle** and an animation that you can play to help visualize the motion of the segments. Pretty neat.

In [ ]:
# Re-use the static plotting function from part 1b. If this cell does not run, make sure you have run the cell that defined that felper function
plot_marker_trajectory(trimmed_data,['RHIP','RLKN','RCAL','RLM','RTOE'])
plt.show()

# Create a helper function to animate the leg segments (hip->knee, knee->ankle, ankle->toe) using either direct marker positions or forward kinematics based on optimized joint angles. Include an option to compare both methods side-by-side when using joint angles.
def animate_leg_segments(
    df=None,
    joint_angle=False,
    joint_dict=None,
    segment_lengths=None,
    interval=50,
    compare_IK_methods=False,
    unconstrained_df=None  # For comparison when using joint_angle=True
):
    """
    Animate leg segments (hip->knee, knee->ankle, ankle->toe).
    """
    
    if compare_IK_methods and joint_angle:
        # Create side-by-side comparison
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
        
        # Setup left plot (direct markers)
        hip_knee1,   = ax1.plot([], [], 'r-', lw=3, label='Thigh')
        knee_ankle1, = ax1.plot([], [], 'g-', lw=3, label='Shank')
        ankle_toe1,  = ax1.plot([], [], 'b-', lw=3, label='Foot')
        ax1.set_title('Direct Markers (Unconstrained)')
        ax1.set_xlabel('Y location (mm)')
        ax1.set_ylabel('Z location (mm)')
        ax1.legend(frameon=False)
        ax1.grid(True, alpha=0.3)
        
        # Setup right plot (forward kinematics)
        hip_knee2,   = ax2.plot([], [], 'r-', lw=3, label='Thigh')
        knee_ankle2, = ax2.plot([], [], 'g-', lw=3, label='Shank')
        ankle_toe2,  = ax2.plot([], [], 'b-', lw=3, label='Foot')
        ax2.set_title('Optimized Angles (Constrained)')
        ax2.set_xlabel('Y location (mm)')
        ax2.set_ylabel('Z location (mm)')
        ax2.legend(frameon=False)
        ax2.grid(True, alpha=0.3)
        
        # Set consistent axis limits
        all_dfs = [unconstrained_df, df] if unconstrained_df is not None else [df]
        all_y_vals = []
        all_z_vals = []
        
        for data_df in all_dfs:
            if data_df is not None:
                all_y_vals.extend(data_df.filter(like='_Y').values.flatten())
                all_z_vals.extend(data_df.filter(like='_Z').values.flatten())
        
        y_min, y_max = min(all_y_vals), max(all_y_vals)
        z_min, z_max = min(all_z_vals), max(all_z_vals)
        
        ax1.set_xlim(y_min - 20, y_max + 20)
        ax1.set_ylim(z_min - 20, z_max + 20)
        ax2.set_xlim(y_min - 20, y_max + 20)
        ax2.set_ylim(z_min - 20, z_max + 20)
        
        n_frames = len(joint_dict['theta_hip'])
        
        def animate_compare(frame):
            # Left plot: Direct markers
            if unconstrained_df is not None:
                pos_direct = {
                    'RHIP': (unconstrained_df.iloc[frame]['RHIP_Y'], unconstrained_df.iloc[frame]['RHIP_Z']),
                    'RLKN': (unconstrained_df.iloc[frame]['RLKN_Y'], unconstrained_df.iloc[frame]['RLKN_Z']),
                    'RLM':  (unconstrained_df.iloc[frame]['RLM_Y'],  unconstrained_df.iloc[frame]['RLM_Z']),
                    'RTOE': (unconstrained_df.iloc[frame]['RTOE_Y'], unconstrained_df.iloc[frame]['RTOE_Z']),
                }
            else:
                pos_direct = {
                    'RHIP': (df.iloc[frame]['RHIP_Y'], df.iloc[frame]['RHIP_Z']),
                    'RLKN': (df.iloc[frame]['RLKN_Y'], df.iloc[frame]['RLKN_Z']),
                    'RLM':  (df.iloc[frame]['RLM_Y'],  df.iloc[frame]['RLM_Z']),
                    'RTOE': (df.iloc[frame]['RTOE_Y'], df.iloc[frame]['RTOE_Z']),
                }
            
            hip_knee1.set_data([pos_direct['RHIP'][0], pos_direct['RLKN'][0]],
                               [pos_direct['RHIP'][1], pos_direct['RLKN'][1]])
            knee_ankle1.set_data([pos_direct['RLKN'][0], pos_direct['RLM'][0]],
                                 [pos_direct['RLKN'][1], pos_direct['RLM'][1]])
            ankle_toe1.set_data([pos_direct['RLM'][0], pos_direct['RTOE'][0]],
                                [pos_direct['RLM'][1], pos_direct['RTOE'][1]])
            
            # Right plot: Forward kinematics
            # Grab global positions from dict, or fallback to the measured hip
            y_glob = joint_dict['y_global'][frame] if 'y_global' in joint_dict else df.iloc[frame]['RHIP_Y']
            z_glob = joint_dict['z_global'][frame] if 'z_global' in joint_dict else df.iloc[frame]['RHIP_Z']
            th_glob = joint_dict['theta_global'][frame] if 'theta_global' in joint_dict else 0.0

            q_frame = [
                y_glob,
                z_glob,
                th_glob,
                joint_dict['theta_hip'][frame],
                joint_dict['theta_knee'][frame],
                joint_dict['theta_ankle'][frame]
            ]
            
            markers_fk = forward_kinematics(q=q_frame, segment_lengths=segment_lengths)
            
            hip_knee2.set_data([markers_fk[0][0], markers_fk[1][0]],
                               [markers_fk[0][1], markers_fk[1][1]])
            knee_ankle2.set_data([markers_fk[1][0], markers_fk[2][0]],
                                 [markers_fk[1][1], markers_fk[2][1]])
            ankle_toe2.set_data([markers_fk[2][0], markers_fk[3][0]],
                                [markers_fk[2][1], markers_fk[3][1]])
            
            return hip_knee1, knee_ankle1, ankle_toe1, hip_knee2, knee_ankle2, ankle_toe2
        
        ani = FuncAnimation(fig, animate_compare, frames=n_frames, 
                            interval=interval, blit=True, repeat=True)
        
    else:
        # Original single animation logic
        fig, ax = plt.subplots(figsize=(5, 5))
        
        # Segment lines
        hip_knee,   = ax.plot([], [], 'r-', lw=3, label='Thigh')
        knee_ankle, = ax.plot([], [], 'g-', lw=3, label='Shank')
        ankle_toe,  = ax.plot([], [], 'b-', lw=3, label='Foot')
        
        ax.set_xlabel('Y location (mm)')
        ax.set_ylabel('Z location (mm)')
        ax.legend(frameon=False)
        ax.grid(True, alpha=0.3)
        
        # Set axis limits
        if df is not None:
            ax.set_xlim(df.filter(like='_Y').min().min() - 20,
                        df.filter(like='_Y').max().max() + 20)
            ax.set_ylim(df.filter(like='_Z').min().min() - 20,
                        df.filter(like='_Z').max().max() + 20)
        else:
            ax.set_xlim(-200, 200)
            ax.set_ylim(-200, 200)
        
        n_frames = len(df) if not joint_angle else len(joint_dict['theta_hip'])
        
        def get_positions(frame):
            if not joint_angle:
                return {
                    'RHIP': (df.iloc[frame]['RHIP_Y'], df.iloc[frame]['RHIP_Z']),
                    'RLKN': (df.iloc[frame]['RLKN_Y'], df.iloc[frame]['RLKN_Z']),
                    'RLM':  (df.iloc[frame]['RLM_Y'],  df.iloc[frame]['RLM_Z']),
                    'RTOE': (df.iloc[frame]['RTOE_Y'], df.iloc[frame]['RTOE_Z']),
                }
            else:
                # Grab global positions from dict, or fallback to the measured hip
                y_glob = joint_dict['y_global'][frame] if 'y_global' in joint_dict else df.iloc[frame]['RHIP_Y']
                z_glob = joint_dict['z_global'][frame] if 'z_global' in joint_dict else df.iloc[frame]['RHIP_Z']
                th_glob = joint_dict['theta_global'][frame] if 'theta_global' in joint_dict else 0.0

                q_frame = [
                    y_glob,
                    z_glob,
                    th_glob,
                    joint_dict['theta_hip'][frame],
                    joint_dict['theta_knee'][frame],
                    joint_dict['theta_ankle'][frame]
                ]
                
                # Call the forward_kinematics function
                markers = forward_kinematics(q=q_frame, segment_lengths=segment_lengths)
                
                # Convert forward kinematics output (4,2) array to dict for animation
                return {
                    'RHIP': markers[0],
                    'RLKN': markers[1],
                    'RLM':  markers[2],
                    'RTOE': markers[3],
                }
        
        def animate(frame):
            pos = get_positions(frame)
            
            hip_knee.set_data([pos['RHIP'][0], pos['RLKN'][0]],
                              [pos['RHIP'][1], pos['RLKN'][1]])
            knee_ankle.set_data([pos['RLKN'][0], pos['RLM'][0]],
                                [pos['RLKN'][1], pos['RLM'][1]])
            ankle_toe.set_data([pos['RLM'][0], pos['RTOE'][0]],
                               [pos['RLM'][1], pos['RTOE'][1]])
            
            return hip_knee, knee_ankle, ankle_toe
        
        ani = FuncAnimation(fig, animate, frames=n_frames, 
                            interval=interval, blit=True, repeat=True)
    
    plt.close(fig)  # Prevents static image display
    return ani


ani = animate_leg_segments(df=trimmed_data, interval=50)
HTML(ani.to_jshtml())   # renders the animation inline

## 1d: Run unconstrained inverse kinematics

Now that you have loaded and trimmed the data, you are now ready to proceed with 2D inverse kinematics! As a reminder, you have time-varying marker data over a single gait cycle for the right leg, and you now want to find joint angles as a function of time (or gait cycle). This is essentially just a matter of trigonometry. To solve for the joint angles at each time step, we must know two things:
1. The (Y,Z) coordinate locations for each end of thigh, shank, and foot segments at each time step
2. The relationship between the joint coordinate locations and the joint angles

While this sounds simple enough, number 2 is a bit tricky, so we first find **segment angles** relative to the fixed horizontal lab reference ($\phi_H, \phi_K, \phi_A$), and then we relate these absolute angles to each other to get relative **joint angles** ($\theta_H, \theta_K, \theta_A$). In cell 1d, you need to define each of these angles and then finish defining the proximal and distal ends of each of the three segments inside the ```calculate_joint_angles``` function before running the cell. In the schematic below, the red, green, and blue rigid bodies represent the thigh, shank, and foot with corresponding lengths $l_T, l_S, l_F$. The subscripts H, K, A for joint angles correspond to hip, knee, and ankle, respectively.

<img src="https://uofi.box.com/shared/static/1epi81ocp72x8yseo7sa813y353he53x.jpg" alt="Joint angle angle definitions" width="700"/>

In [ ]:
def calculate_joint_angles(marker_positions_df):
    """
    Analystical inverse kinematics to compute joint angles given marker positions.
    
    Parameters:
    -----------
    marker_positions_df (DataFrame): 
        DataFrame containing columns: 'hip_y', 'hip_z', 'knee_y', 'knee_z', 'ankle_y', 'ankle_z', 'toe_y', 'toe_z'
    
    Note:
    ------------
    Units are assumed to be in mm
    
    Returns:
    --------
    angles (dict):
        dictionary containing joint angles (theta_hip, theta_knee, theta_ankle) in radians
    """

    # If input is Series (1D), convert to DataFrame with one row
    if isinstance(marker_positions_df, pd.Series):
        marker_positions_df = marker_positions_df.to_frame().T

    # Step 1: Specify the proximal and distal segment endpoints
    thigh_prox_y = #TODO fill in the proximal thigh Y position using the appropriate column from the marker_positions_df
    thigh_prox_z = #TODO fill in the proximal thigh Z position using the appropriate column from the marker_positions_df
    shank_prox_y = #TODO fill in the proximal shank Y position using the appropriate column from the marker_positions_df
    shank_prox_z = #TODO fill in the proximal shank Z position using the appropriate column from the marker_positions_df
    foot_prox_y = #TODO fill in the proximal foot Y position using the appropriate column from the marker_positions_df
    foot_prox_z = #TODO fill in the proximal foot Z position using the appropriate column from the marker_positions_df

    thigh_dist_y = #TODO fill in the distal thigh Y position using the appropriate column from the marker_positions_df
    thigh_dist_z = #TODO fill in the distal thigh Z position using the appropriate column from the marker_positions_df
    shank_dist_y = #TODO fill in the distal shank Y position using the appropriate column from the marker_positions_df
    shank_dist_z = #TODO fill in the distal shank Z position using the appropriate column from the marker_positions_df
    foot_dist_y = #TODO fill in the distal foot Y position using the appropriate column from the marker_positions_df
    foot_dist_z = #TODO fill in the distal foot Z position using the appropriate column from the marker_positions_df


    # Step 2: Compute the joint angles relative to the vertical axis (lab reference frame) using arctan2 function
    phi_H = #TODO compute the hip angle relative to vertical using the appropriate proximal and distal thigh positions and the arctan2 function
    phi_K = #TODO compute the knee angle relative to vertical using the appropriate proximal and distal shank positions and the arctan2 function
    phi_A = #TODO compute the ankle angle relative to vertical using the appropriate proximal and distal foot positions and the arctan2 function


    # Step 3: Compute the joint angles based on the provided joint angle conventions in the schematic
    theta_H = #TODO compute the hip angle based on the provided joint angle convention schematic above
    theta_K = #TODO compute the knee angle based on the provided joint angle convention schematic above
    theta_A = #TODO compute the ankle angle based on the provided joint angle convention schematic above


    # Create dictionary to hold angles (there is nothing you need to modify below this line in the function)
    joint_angles = {
        'theta_hip': theta_H,
        'theta_knee': theta_K,
        'theta_ankle': theta_A
    }

    return joint_angles

# Helper function to plot joint angles over the gait cycle (nothing you need to modify in this function)
def plot_joint_angles(angles):
    """
    
    Plot joint angles in degrees as a function of percent gait cycle.
    
    Parameters:
    -----------
    angles : dict
        dictionary containing joint angles (theta_hip, theta_knee, theta_ankle) in radians
    
    Returns:
    --------
    None, but displays plots.

    """

    fig, axs = plt.subplots(3, 1, figsize=(8, 12), sharex=True)
    gait_cycle=np.linspace(0, 100, len(angles['theta_hip']))

    # Convert to degrees for plotting
    angles_local = angles.copy()
    for joint in angles_local:
        angles_local[joint] = np.degrees(angles_local[joint])
    axs[0].plot(gait_cycle, angles_local['theta_hip'], label='Hip Angle', color='k')
    axs[0].set_ylabel('Angle (degrees)')
    axs[0].set_title('Hip Joint Angle vs. Gait Cycle (%)')
    axs[1].plot(gait_cycle, angles_local['theta_knee'], label='Knee Angle', color='k')
    axs[1].set_ylabel('Angle (degrees)')
    axs[1].set_title('Knee Joint Angle vs. Gait Cycle (%)')
    axs[2].plot(gait_cycle, angles_local['theta_ankle'], label='Ankle Angle', color='k')
    axs[2].set_ylabel('Angle (degrees)')
    axs[2].set_title('Ankle Joint Angle vs. Gait Cycle (%)')
    axs[2].set_xlabel('Gait Cycle (%)')
    axs[0].grid(alpha=0.3)
    axs[1].grid(alpha=0.3)
    axs[2].grid(alpha=0.3)

    fig.show()
    
    return None

## 1d: Calculate and plot joint angles for the gait cycle of interest

There is nothing you need to change in cell 1d before running it.

In [ ]:
# Use inverse kinematics function to compute joint angles from marker positions
angles = calculate_joint_angles(trimmed_data) # Here we pass in the DataFrame that only contains marker locations for a single right leg gait cycle, which you created in step 1c

# Use the provided plotting function to visualize joint angles vs. percent gait cycle
plot_joint_angles(angles)

# Problem 2: Constrained IK

Unfortunately, the approach we just employed falls short when dealing with 3D motion capture data. Optimization-based inverse kinematics becomes necessary for 3D motion capture because marker data are...
1. Redundant (a single segment only requires 3 markers to be fully constrained, but often 4+ are used)
2. Noisy
3. Inconsistent due to soft tissure artifact
4. Missing for some time steps due to occlusion (marker is hidden and therefore will briefly drop out)

All of this means that there is generally no single exact kinematic configuration that satisfies all marker positions simultaneously. Optimization provides a way to estimate the most *plausible* pose by trading off marker errors, enforcing anatomical constraints, and weighting markers according to reliability. In general, using a musculoskeletal model that incorporates knowledge about the subject's geometry and mobility will produce more accurate estimates of joint angles when compared to the unconstrained kinematics approach.

Here is the mathematical basis of our problem: Determine the generalized coordinates $\underline{q}=[y_{global}, z_{global}, \theta_{global}, \theta_{hip}, \theta_{knee}, \theta_{ankle}]$ that minimize the error between predicted marker positions (from forward kinematics) and measured noisy marker positions. Forward kinematics is used to predict marker locations given the global position and orientation of the model, segment lengths, and relative segment angles.

**Parameters to optimize:**
- The generalized coordinates at each time point (or frame), $\underline{q}=[y_{global}, z_{global}, \theta_{global}, \theta_{hip}, \theta_{knee}, \theta_{ankle}]$
    - $y_{global}, z_{global}$ denote global translations of the multi-segment model (i.e. shift the entire model around in space)
    - $\theta_{global}$ denotes global rotation of the of the multi-segment model (i.e. rotate the entire model)
    - $\theta_{hip}, \theta_{knee}, \theta_{ankle}$ denote joint angles of the multi-segment model (i.e. modify the anatomical joint angles)

**Objective function:**
- The sum of squared errors between predicted and measured marker positions for all markers (hip, knee, ankle, toe): $\Sigma_{k \in Markers} \vert \vert \underline{x}^{exp}_k - \underline{x}_k(\underline{q}) \vert \vert^2 $. Here, $\underline{x}^{exp}_k$ represents the experimentally measured $y,z$ location of the $k^{th}$ marker and $\underline{x}_k(\underline{q})$ represents the predicted $y,z$ location of the $k^{th}$ marker based on the current state of the musculoskeletal model. Recall that we are only concerned with $y$ and $z$ coordinates because we are making a simplifying assumption of planar motion.

**Constraints:**
- Joint angle limits (physiologically feasible values). Recall this is not strictly necessary for constrained IK, but practically it is helpful for convergence since it limits the total possible configurations that the multi-segment rigid body model may assume.
- Fixed segment lengths (thigh, shank, foot).
- All joints here are assumed to behave as pin joints.

## 2a: Intentionally add a bias to our knee marker position

Let's look at a simple case where constrained IK is more resistant to marker bias (i.e. a misplaced marker relative to the true anatomical location of interest). Constrained IK really shines (or unconstrained IK struggles...) with more redundant markers and dealing with 3D rigid body motion, but for the sake of simplicity so you understand the underlying algoriythm and logic, we will not introduce all of that in this problem (breath a sigh of relief). Instead, using the same dataset that we loaded for problem 1, let us imagine that the person who applied the marker to the right lateral knee (RLKN) placed the marker 35 mm too far proximally on the thigh segment (i.e. in the direction of right hip). The function `bias_marker_along_segment` accomplishes this for the data we were working with in problem 1. There is nothing you need to change in cell 2a. Just run the cell.

In [ ]:
def bias_marker_along_segment(marker_df, target_marker, reference_marker, bias_amount):
    """
    Shifts a target marker along the vector pointing towards a reference marker.
    This simulates placing a marker too high/low along a specific body segment.
    
    Parameters:
    -----------
    marker_df : DataFrame
        Marker positions (columns: e.g., 'RHIP_Y', 'RHIP_Z')
    target_marker : str
        The marker to move (e.g., 'RLKN')
    reference_marker : str
        The marker to move TOWARDS (e.g., 'RHIP')
    bias_amount : float
        Distance to move the target marker in mm (positive = towards reference)
        
    Returns:
    --------
    biased_df : DataFrame
        Marker positions with the systematic bias applied
    """
    biased_df = marker_df.copy()
    
    # Define the column names based on the marker strings
    target_y = f"{target_marker}_Y"
    target_z = f"{target_marker}_Z"
    ref_y = f"{reference_marker}_Y"
    ref_z = f"{reference_marker}_Z"
    
    # 1. Calculate the vector from target to reference for all frames simultaneously
    vec_y = biased_df[ref_y] - biased_df[target_y]
    vec_z = biased_df[ref_z] - biased_df[target_z]
    
    # 2. Calculate the magnitude (length) of the vector at each frame
    # Using the standard Euclidean distance: $\sqrt{y^2 + z^2}$
    magnitude = np.sqrt(vec_y**2 + vec_z**2)
    
    # 3. Calculate the normalized vector (unit vector) components
    unit_y = vec_y / magnitude
    unit_z = vec_z / magnitude
    
    # 4. Apply the bias along that vector
    biased_df[target_y] += unit_y * bias_amount
    biased_df[target_z] += unit_z * bias_amount
    
    return biased_df

# Simulate placing the knee marker 35 mm too high up the thigh towards the hip
biased_trimmed_data = bias_marker_along_segment(
    marker_df=trimmed_data, 
    target_marker='RLKN', 
    reference_marker='RHIP', 
    bias_amount=35.0
)

print("A 35 mm systematic bias has been applied to the right knee marker, shifting it towards the hip.")

## 2b: Define forward kinematics function and objective function for optimization

In the background for this problem, we explained that we will be using optimization to minimize an error term that compares marker locations from our muskuloskeletal model, global translations and rotation, and joint angles (forward kinematics) and experimentally measured markers. 

- ```forward_kinematics``` defines all predicted marker locations given the rigid body model (i.e. thigh, shank, foot segments of known length connected via pin joints), and generalized coordinate vector $\underline{q}=[y_{global}, z_{global}, \theta_{global}, \theta_{hip}, \theta_{knee}, \theta_{ankle}]$.
- ```objective_function``` defines an error term that is to be minimized given the rigid body model (i.e. thigh, shank, foot segments of known length connected via pin joints), and generalized coordinate vector $\underline{q}=[y_{global}, z_{global}, \theta_{global}, \theta_{hip}, \theta_{knee}, \theta_{ankle}]$. Notice that it actually calls the ```forward_kinematics``` function.

There is nothing you need to change in cell 2b. Skim the ```forward_kinematics``` and ```objective_function``` functions to understand the input arguments and their outputs before running the cell.

In [ ]:
# 1. Define the updated forward kinematics using the generalized coordinate, q = [y_global, z_global, theta_global, theta_hip, theta_knee, theta_ankle]
def forward_kinematics(q, segment_lengths):
    """
    Compute marker positions from generalized coordinates and segment lengths.
    
    Parameters:
    -----------
    q : array-like, shape (6,)
        Generalized coordinates [y_global, z_global, theta_global, theta_hip, theta_knee, theta_ankle]
    segment_lengths : dict
        {'thigh': float, 'shank': float, 'foot': float}
        
    Returns:
    --------
    markers_pred : np.array shape (4, 2)
        Predicted marker positions: hip, knee, ankle, toe (y,z)
    """
    # Unpack the generalized coordinates
    y_global, z_global, theta_global, theta_hip, theta_knee, theta_ankle = q
    
    thigh_len = segment_lengths['thigh']
    shank_len = segment_lengths['shank']
    foot_len = segment_lengths['foot']
    
    # 1. Global Translation defines the Hip position
    hip = np.array([y_global, z_global])
    
    # 2. Calculate absolute angles for each segment
    # Assuming theta_global acts as the base pelvis/trunk orientation
    abs_thigh_angle = theta_global + theta_hip
    abs_shank_angle = abs_thigh_angle - theta_knee
    abs_foot_angle = abs_shank_angle + theta_ankle
    
    # 3. Build the rigid body segments
    knee = hip + thigh_len * np.array([np.sin(abs_thigh_angle), -np.cos(abs_thigh_angle)])
    ankle = knee + shank_len * np.array([np.sin(abs_shank_angle), -np.cos(abs_shank_angle)])
    toe = ankle + foot_len * np.array([np.cos(abs_foot_angle), np.sin(abs_foot_angle)])
    
    markers_pred = np.vstack([hip, knee, ankle, toe])
    return markers_pred

# 2. Define the objective function that is minimized during optimization (sum of squared marker position errors)
def objective_function(q, markers_measured, segment_lengths):
    """
    Objective function to minimize: sum of squared marker position errors.
    
    Parameters:
    -----------
    q : array-like, shape (6,)
        Generalized coordinates
    markers_measured : np.array, shape (4, 2)
        Measured marker positions (y,z) for hip, knee, ankle, toe
    segment_lengths : dict
        Segment lengths
        
    Returns:
    --------
    error : float
        Sum of squared errors
    """
    # Generate the model's guess of where the markers are
    markers_pred = forward_kinematics(q, segment_lengths)
    
    # Calculate the sum of squared errors
    error = np.sum((markers_pred - markers_measured)**2)
    
    return error

# 3. Define an optimization "wrapper" that calls scipy's minimize function to solve for the optimal generalized coordinates given marker measurements and segment lengths
def optimize_kinematics(markers_measured, segment_lengths, initial_guess=None, bounds=None):
    """
    Solves for the optimal generalized coordinates for a single frame of data.
    """
    if initial_guess is None:
        # This is our generalized coordinate vector, q = [y_glob, z_glob, th_glob, th_hip, th_knee, th_ankle]
        initial_guess = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0]) 
        
    # Seed the global translation guess with the measured hip marker to help with convergence 
    initial_guess[0] = markers_measured[0, 0] # y_global guess
    initial_guess[1] = markers_measured[0, 1] # z_global guess
    
    result = minimize(objective_function, initial_guess,
                      args=(markers_measured, segment_lengths),
                      bounds=bounds,
                      method='L-BFGS-B')
    
    return result.x  # Returns the optimized generalized coordinates

## 2c: Define segment lengths, reasonable bounds, and solve optimization problem

In this step, we will actually perform the least squares optimization frame-by-frame over the same gait cycle of interest from problem 1. We provide the optimizer bounds on the generalized coordinates that will be input into the `forward_kinematics` function. **You will need to set these bounds.** 

Since we are using a floating-base model, our state vector, $q$, has 6 variables: $[y_{global}, z_{global}, \theta_{global}, \theta_{hip}, \theta_{knee}, \theta_{ankle}]$. You will leave the global translation bounds relatively unconstrained (set them to `None`), but you need to determine the bounds for the three relative joint angles and set bounds for the global rotation angle. Recall that this is the same data that you plotted using the analytical inverse kinematics in problem 1, so you should have a good idea of reasonable physiological bounds from the plot that you generated at the end of problem 1.

You are also passing fixed segment lengths into the `forward_kinematics` function, but we have calculated what these should be given the *clean* marker data (e.g., from frame 0 **before** we added a position bias to the knee marker) for you.

Here is, in a nutshell, what is going on with this optimization routine:
 - **The Initial Guess:** The optimizer starts with a guess for the state vector $q$. To save time and ensure physical realism, it uses the optimized result from the previous frame (since the body doesn't move much in a fraction of a second).
 - **Forward Kinematics:** It plugs that $q$ into the forward_kinematics function, which uses your fixed segment lengths to calculate exactly where the hip, knee, ankle, and toe markers would be in space for those specific angles.
 - **The Error Calculation:** It compares those predicted marker positions to the measured (noisy) marker data. The "cost" is the sum of the squared distances between them. 
 $Error=\Sigma_{k \in Markers} \vert \vert \underline{x}^{exp}_k - \underline{x}_k(\underline{q}) \vert \vert^2 $. Here, $\underline{x}^{exp}_k$ represents the experimentally measured location of the $k^{th}$ marker and $\underline{x}_k(\underline{q})$ represents the predicted location of the $k^{th}$ marker based on the current state of the musculoskeletal model.
 - **Adjustment:** If the error is high, the optimizer nudges the values in $q$ (within your defined bounds) and repeats the last two steps.
 - **The Solution:** It stops when it finds the specific $q$ that makes the predicted markers align as closely as possible to the experimental markers.

In [ ]:
# Define segment lengths using the first frame of downsampled data (Use clean data, not noisy data!)

segment_lengths = {
    'thigh': np.linalg.norm(trimmed_data[['RLKN_Y', 'RLKN_Z']].iloc[0].values - trimmed_data[['RHIP_Y', 'RHIP_Z']].iloc[0].values),
    'shank': np.linalg.norm(trimmed_data[['RLM_Y', 'RLM_Z']].iloc[0].values - trimmed_data[['RLKN_Y', 'RLKN_Z']].iloc[0].values),
    'foot': np.linalg.norm(trimmed_data[['RTOE_Y', 'RTOE_Z']].iloc[0].values - trimmed_data[['RLM_Y', 'RLM_Z']].iloc[0].values)
}

# Define realistic bounds for our 6 generalized coordinates: 
# [y_glob, z_glob, th_glob, th_hip, th_knee, th_ankle]
# (None, None) means the optimizer has no limits for that specific variable
bounds = [
   (#TODO fill in lower bound, #TODO fill in upper bound),  # y_global bounds in mm
   (#TODO fill in lower bound, #TODO fill in upper bound),  # z_global bounds in mm
   (np.deg2rad(#TODO fill in lower bound), np.deg2rad(#TODO fill in upper bound)),  # theta_global bounds in degrees, use your intuition here about how much the pelvis/trunk might rotate in the sagittal plane during normal walking (hint: it's not that much, but you can experiment with different bounds to see how it affects the results!)
   (np.deg2rad(#TODO fill in lower bound), np.deg2rad(#TODO fill in upper bound)),  # theta_hip in degrees, use your plot from problem 1 to help help set these bounds
   (np.deg2rad(#TODO fill in lower bound), np.deg2rad(#TODO fill in upper bound)),  # theta_knee in degrees, use your plot from problem 1 to help help set these bounds
   (np.deg2rad(#TODO fill in lower bound), np.deg2rad(#TODO fill in upper bound))   # theta_ankle in degrees, use your plot from problem 1 to help help set these bounds
]

#There is nothing beneath this line that you need to modify, but this is where we will run the optimization for each frame of our biased data and store the results in a dictionary for plotting later
# Update dictionary to store all 6 coordinates
optimized_q = {
    'y_global': [], 'z_global': [], 'theta_global': [], 
    'theta_hip': [], 'theta_knee': [], 'theta_ankle': []
}

# For first frame, we will build an initial guess
prev_solution = None

# Note: Using biased_trimmed_data based on your previous step
for i in range(len(biased_trimmed_data)): 
    # Extract measured markers for this frame
    markers_measured = biased_trimmed_data.iloc[i][[
        'RHIP_Y', 'RHIP_Z',
        'RLKN_Y', 'RLKN_Z',
        'RLM_Y', 'RLM_Z',
        'RTOE_Y', 'RTOE_Z'
    ]].values.reshape(4, 2)

    if prev_solution is None:
        # Calculate unconstrained angles just to seed the first frame
        direct_ik_angles = calculate_joint_angles(biased_trimmed_data.iloc[[i]][['RHIP_Y', 'RHIP_Z', 'RLKN_Y', 'RLKN_Z', 'RLM_Y', 'RLM_Z', 'RTOE_Y', 'RTOE_Z']])
        
        # Build the generalized coordinate initial guess
        initial_guess = np.array([
            markers_measured[0, 0],                     # y_global guess (measured hip y)
            markers_measured[0, 1],                     # z_global guess (measured hip z)
            0.0,                                        # theta_global guess
            direct_ik_angles['theta_hip'].values[0],    # th_hip guess
            direct_ik_angles['theta_knee'].values[0],   # th_knee guess
            direct_ik_angles['theta_ankle'].values[0]   # th_ankle guess
        ])
    else:
        # Use previous frame's full 6-DoF solution
        initial_guess = prev_solution.copy()
        
        # Overwrite the global translation guess with the current frame's measured hip.
        initial_guess[0] = markers_measured[0, 0]
        initial_guess[1] = markers_measured[0, 1]

    # Run the optimization wrapper we defined in 2b
    opt_q = optimize_kinematics(markers_measured, segment_lengths, initial_guess, bounds=bounds)

    # Update previous solution for the next loop iteration
    prev_solution = opt_q

    # Store results
    optimized_q['y_global'].append(opt_q[0])
    optimized_q['z_global'].append(opt_q[1])
    optimized_q['theta_global'].append(opt_q[2])
    optimized_q['theta_hip'].append(opt_q[3])
    optimized_q['theta_knee'].append(opt_q[4])
    optimized_q['theta_ankle'].append(opt_q[5])

# Convert lists to numpy arrays for easier plotting later
for key in optimized_q:
    optimized_q[key] = np.array(optimized_q[key])

## 2d: Compare unconstrained IK and constrained IK results

There is nothing you need to change in cell 2d. Just run the cell and observe the differences between the unconstrained inverse kinematics results

In [ ]:
def plot_comparison(direct_angles, optimized_angles):
    fig, axs = plt.subplots(3, 1, figsize=(8, 12), sharex=True)
    gait_cycle = np.linspace(0, 100, len(direct_angles['theta_hip']))
    
    for joint in ['theta_hip', 'theta_knee', 'theta_ankle']:
        axs[['theta_hip', 'theta_knee', 'theta_ankle'].index(joint)].plot(gait_cycle, np.degrees(direct_angles[joint]), label='Direct IK', color='blue')
        axs[['theta_hip', 'theta_knee', 'theta_ankle'].index(joint)].plot(gait_cycle, np.degrees(optimized_angles[joint]), label='Optimization IK', color='red', linestyle='--')
        axs[['theta_hip', 'theta_knee', 'theta_ankle'].index(joint)].set_ylabel('Angle (degrees)')
        axs[['theta_hip', 'theta_knee', 'theta_ankle'].index(joint)].set_title(f'{joint.replace("_", " ").title()} vs. Gait Cycle (%)')
        axs[['theta_hip', 'theta_knee', 'theta_ankle'].index(joint)].legend()
        axs[['theta_hip', 'theta_knee', 'theta_ankle'].index(joint)].grid(alpha=0.3)
    
    axs[2].set_xlabel('Gait Cycle (%)')
    plt.show()

# Call plot function
plot_comparison(calculate_joint_angles(biased_trimmed_data), optimized_q)


# Also create animation of the segments using constrained and unconstrained IK results
ani_constrained = animate_leg_segments(
    df=biased_trimmed_data,
    joint_angle=True,
    joint_dict=optimized_q,
    segment_lengths=segment_lengths,
    compare_IK_methods=True,
    unconstrained_df=biased_trimmed_data,
    interval=50
)

HTML(ani_constrained.to_jshtml())   # renders the animation inline

## Notebook export

Export your notebook as an HTML for upload to Canvas.

In [ ]:
# Mount Google Drive - you can also do this manually by clicking on the folder icon on the left sidebar and then clicking "Mount Drive"
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import nbformat
from nbconvert import HTMLExporter
from google.colab import files

# Helper function to export a Colab notebook as an HTML file
def export_current_notebook(notebook_path, output_filename="exported_notebook.html"):
    """
    Exports the specified Colab notebook (with all cell outputs) as an HTML file.
    """
    try:
        # Convert to HTML
        with open(notebook_path, "r", encoding="utf-8") as f:
            notebook_content = nbformat.read(f, as_version=4)

        # Convert notebook to HTML
        html_exporter = HTMLExporter()
        html_exporter.exclude_input = False
        html_exporter.exclude_output = False
        html_data, _ = html_exporter.from_notebook_node(notebook_content)

        # Add CSS to wrap lines and prevent clipping
        wrap_css = """
        <style>
            pre, code {
                word-wrap: break-word;
                white-space: pre-wrap;
                word-break: break-word;
            }
        </style>
        """
        html_data = wrap_css + html_data

        # Save the HTML file
        with open(output_filename, "w", encoding="utf-8") as f:
            f.write(html_data)

        print(f"Notebook exported successfully as {output_filename}")

        # Optionally download the file
        files.download(output_filename)

    except Exception as e:
        print(f"An error occurred: {e}")


# Change to your notebook path
notebook_to_export = #TODO: Copy the path to this notebook from the file tree and paste it here as a string
# This will save to your local downloads
output_filename = 'ME481-HW3-IK.html'

export_current_notebook(notebook_path=notebook_to_export,
                        output_filename=output_filename)